In [ ]:
# ==========================================
# 0. MOUNT GOOGLE DRIVE
# ==========================================
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive berhasil disambungkan.")
except ImportError:
    print("Tidak berjalan di Colab.")

# ==========================================
# 1. IMPORT LIBRARIES
# ==========================================
import torch
import torch.nn as nn
from torchvision import models
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset
from torchvision import transforms
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
import torch.nn.functional as F
import pandas as pd
import numpy as np

# ==========================================
# 2. KONFIGURASI FOLDER & DEVICE
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Menggunakan device: {device}")

# Path ke folder VGG19 di Drive kamu
FOLDER_SAMPLER = '/content/drive/MyDrive/Eksperimen_Alzheimer_VGG19'
FOLDER_CLASS_WEIGHT = '/content/drive/MyDrive/Eksperimen_Alzheimer_VGG19_CW'

# ==========================================
# 3. DEFINISI CLASS DATASET
# ==========================================
class AlzheimerDataset(Dataset):
    def __init__(self, hf_data, transform=None):
        self.data = hf_data
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        image = item["image"].convert("RGB")
        label = item["label"]

        if self.transform:
            image = self.transform(image)

        return image, label

# ==========================================
# 4. PERSIAPAN DATA UJI (TEST DATASET)
# ==========================================
print("\nMenyiapkan Data Test...")
hf_dataset = load_dataset("Falah/Alzheimer_MRI")
test_data = hf_dataset["test"]

# VGG19 membutuhkan input resolusi 224x224
base_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

test_ds = AlzheimerDataset(test_data, base_transform)
test_loader = DataLoader(test_ds, batch_size=16, shuffle=False, num_workers=2)

# ==========================================
# 5. INISIALISASI ARSITEKTUR MODEL VGG19
# ==========================================
print("\nMembangun arsitektur VGG19...")
model = models.vgg19(weights=None)

num_classes = 4

# Custom classifier VGG19 sesuai dengan skrip training
model.classifier = nn.Sequential(
    nn.Linear(25088, 4096),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(4096, 1024),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(1024, num_classes)
)
model = model.to(device)
model.eval()

# ==========================================
# 6. FUNGSI EVALUASI MASSAL
# ==========================================
def evaluate_models_in_folder(folder_path, method_name):
    results = []

    if not os.path.exists(folder_path):
        print(f"Folder tidak ditemukan: {folder_path}")
        return results

    model_files = [f for f in os.listdir(folder_path) if f.endswith('.pth')]
    model_files.sort()

    for filename in model_files:
        filepath = os.path.join(folder_path, filename)

        # Logika Parsing File Name VGG19
        # Contoh: best_vgg19_aug_sampler_full_tuning_adamw_cosine.pth
        clean_name = filename.replace('best_vgg19_aug_sampler_', '').replace('best_vgg19_aug_cw_', '').replace('.pth', '')
        parts = clean_name.split('_')

        try:
            # Ambil elemen dari belakang karena level fine_tune bisa mengandung underscore (full_tuning)
            scheduler = parts[-1]
            optimizer = parts[-2]
            scenario = "_".join(parts[:-2])
        except Exception:
            scenario, optimizer, scheduler = filename, "-", "-"

        # 1. LOAD WEIGHTS
        model.load_state_dict(torch.load(filepath, map_location=device))

        # 2. PROSES INFERENCE DENGAN PROBABILITAS
        all_labels = []
        all_preds = []
        all_probs = []

        with torch.no_grad():
            for images, labels in test_loader:
                images = images.to(device)
                outputs = model(images)

                probs = F.softmax(outputs, dim=1)
                preds = torch.argmax(probs, dim=1)

                all_labels.extend(labels.cpu().numpy())
                all_preds.extend(preds.cpu().numpy())
                all_probs.extend(probs.cpu().numpy())

        # 3. HITUNG METRIK
        acc = accuracy_score(all_labels, all_preds)
        precision, recall, f1, _ = precision_recall_fscore_support(
            all_labels, all_preds, average='macro', zero_division=0
        )
        roc_auc = roc_auc_score(all_labels, all_probs, multi_class='ovr', average='macro')

        # 4. SIMPAN HASIL
        results.append({
            "Method": method_name,
            "Scenario (Tuning)": scenario.upper(),
            "Opt": optimizer.upper(),
            "Sched": scheduler.upper(),
            "Accuracy": acc,
            "F1-Macro": f1,
            "Precision": precision,
            "Recall": recall,
            "ROC-AUC": roc_auc
        })

    return results

# ==========================================
# 7. JALANKAN EKSEKUSI & TAMPILKAN LAPORAN
# ==========================================
all_results = []

# Evaluasi folder Sampler
print(f"\n--- Memulai Evaluasi Folder: Sampler ---")
sampler_results = evaluate_models_in_folder(FOLDER_SAMPLER, "Sampler")
all_results.extend(sampler_results)

# Evaluasi folder Class Weight
print(f"\n--- Memulai Evaluasi Folder: Class Weight ---")
cw_results = evaluate_models_in_folder(FOLDER_CLASS_WEIGHT, "Class Weight")
all_results.extend(cw_results)

if len(all_results) > 0:
    # Konversi ke DataFrame Pandas
    df_results = pd.DataFrame(all_results)

    # URUTKAN BERDASARKAN F1-MACRO & ROC-AUC TERTINGGI
    df_results_sorted = df_results.sort_values(by=['F1-Macro', 'ROC-AUC'], ascending=[False, False])

    print("\n\n================ KESIMPULAN HASIL EVALUASI VGG19 ================")

    # SETTING PRESISI PANDAS KE 4 ANGKA DESIMAL (f4) & TAMPILKAN SEMUA BARIS
    pd.set_option('display.float_format', '{:.4f}'.format)
    pd.set_option('display.max_rows', None)

    display(df_results_sorted)

    # (Opsional) Simpan laporan ke CSV di Drive
    # df_results_sorted.to_csv('/content/drive/MyDrive/Laporan_Evaluasi_VGG19.csv', index=False)
else:
    print("\nTidak ada model yang berhasil dievaluasi. Periksa kembali path folder Drive kamu.")

Mounted at /content/drive
Google Drive berhasil disambungkan.
Menggunakan device: cuda

Menyiapkan Data Test...


README.md:   0%|          | 0.00/2.13k [00:00<?, ?B/s]

data/train-00000-of-00001-c08a401c53fe53(…):   0%|          | 0.00/22.6M [00:00<?, ?B/s]

data/test-00000-of-00001-44110b9df98c558(…):   0%|          | 0.00/5.65M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5120 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1280 [00:00<?, ? examples/s]


Membangun arsitektur VGG19...

--- Memulai Evaluasi Folder: Sampler ---

--- Memulai Evaluasi Folder: Class Weight ---


================ KESIMPULAN HASIL EVALUASI VGG19 ================


,Method,Scenario (Tuning),Opt,Sched,Accuracy,F1-Macro,Precision,Recall,ROC-AUC
9,Sampler,FULL_TUNING_ADAMW,REDUCE,LR,0.9672,0.9716,0.9684,0.9750,0.9982
8,Sampler,FULL_TUNING,ADAMW,COSINE,0.9641,0.9711,0.9689,0.9734,0.9962
7,Sampler,FULL_TUNING_ADAM,REDUCE,LR,0.9734,0.9706,0.9816,0.9603,0.9971
6,Sampler,FULL_TUNING,ADAM,COSINE,0.9664,0.9645,0.9800,0.9504,0.9962
27,Class Weight,FULL_TUNING_ADAMW,REDUCE,LR,0.9609,0.9616,0.9705,0.9533,0.9970
24,Class Weight,FULL_TUNING,ADAM,COSINE,0.9555,0.9506,0.9501,0.9514,0.9946
25,Class Weight,FULL_TUNING_ADAM,REDUCE,LR,0.9609,0.9499,0.9687,0.9335,0.9959
13,Sampler,UNFREEZE_LAST_ADAM,REDUCE,LR,0.9328,0.9417,0.9508,0.9335,0.9905
26,Class Weight,FULL_TUNING,ADAMW,COSINE,0.9297,0.9387,0.9425,0.9363,0.9924
30,Class Weight,UNFREEZE_LAST,ADAM,COSINE,0.9273,0.9340,0.9408,0.9279,0.9852


In [ ]:
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.preprocessing import label_binarize
import numpy as np
import torch
import torch.nn.functional as F

# Pastikan list kelas ini sesuai dengan urutan index dataset (0, 1, 2, 3)
class_names = ["Mild Demented", "Moderate Demented", "Non Demented", "Very Mild Demented"]

def evaluate_and_visualize_all(folder_path, method_name):
    if not os.path.exists(folder_path):
        print(f"Folder tidak ditemukan: {folder_path}")
        return

    model_files = [f for f in os.listdir(folder_path) if f.endswith('.pth')]

    if len(model_files) == 0:
        print(f"Folder {method_name} kosong. Tidak ada file .pth ditemukan.")
        return

    model_files.sort() # Urutkan file model

    for filename in model_files:
        filepath = os.path.join(folder_path, filename)

        # Header pemisah antar model agar outputnya rapi
        print("\n" + "="*80)
        print(f"🚀 METODE: {method_name.upper()} | MODEL: {filename}")
        print("="*80)

        # 1. LOAD WEIGHTS
        model.load_state_dict(torch.load(filepath, map_location=device))
        model.eval() # Pastikan model dalam mode evaluasi

        # 2. INFERENCE
        all_labels = []
        all_preds = []
        all_probs = []

        with torch.no_grad():
            # Dataset VGG19 di-unpack sebagai (images, labels)
            for images, labels in test_loader:
                images = images.to(device)
                outputs = model(images)

                probs = F.softmax(outputs, dim=1)
                preds = torch.argmax(probs, dim=1)

                all_labels.extend(labels.cpu().numpy())
                all_preds.extend(preds.cpu().numpy())
                all_probs.extend(probs.cpu().numpy())

        all_labels = np.array(all_labels)
        all_preds = np.array(all_preds)
        all_probs = np.array(all_probs)

        # 3. CLASSIFICATION REPORT
        print("\n[1] CLASSIFICATION REPORT")
        print(classification_report(all_labels, all_preds, target_names=class_names, digits=4, zero_division=0))

        # 4. CONFUSION MATRIX & ROC CURVE
        print("\n[2] CONFUSION MATRIX & ROC CURVE")
        fig, axes = plt.subplots(1, 2, figsize=(16, 6)) # Jejerkan 2 grafik kiri-kanan

        # Membersihkan nama file panjang untuk judul grafik agar lebih rapi
        clean_title = filename.replace('best_vgg19_aug_sampler_', '').replace('best_vgg19_aug_cw_', '').replace('.pth', '')

        # -- Grafik Kiri: Confusion Matrix --
        cm = confusion_matrix(all_labels, all_preds)
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                    xticklabels=class_names, yticklabels=class_names, ax=axes[0])
        axes[0].set_xlabel("Predicted", fontweight='bold')
        axes[0].set_ylabel("Actual", fontweight='bold')
        axes[0].set_title(f"CM: {clean_title.upper()}", fontweight='bold')
        axes[0].tick_params(axis='x', rotation=45)

        # -- Grafik Kanan: ROC Curve --
        y_bin = label_binarize(all_labels, classes=[0, 1, 2, 3])
        colors = ['blue', 'red', 'green', 'orange']

        for i, color in zip(range(4), colors):
            fpr, tpr, _ = roc_curve(y_bin[:, i], all_probs[:, i])
            roc_auc = auc(fpr, tpr)
            axes[1].plot(fpr, tpr, color=color, lw=2,
                         label=f'{class_names[i]} (AUC = {roc_auc:.4f})')

        axes[1].plot([0, 1], [0, 1], 'k--', lw=2)
        axes[1].set_xlim([0.0, 1.0])
        axes[1].set_ylim([0.0, 1.05])
        axes[1].set_xlabel('False Positive Rate', fontweight='bold')
        axes[1].set_ylabel('True Positive Rate', fontweight='bold')
        axes[1].set_title('ROC Curve', fontweight='bold')
        axes[1].legend(loc="lower right")
        axes[1].grid(alpha=0.3)

        plt.tight_layout()
        plt.show() # Tampilkan gambar di Colab

        # SANGAT PENTING: Bebaskan memori Matplotlib setelah ditampilkan!
        plt.close('all')

# ==========================================
# EKSEKUSI UNTUK KEDUA FOLDER VGG19
# ==========================================
# Path folder disesuaikan dengan folder VGG19 milikmu
FOLDER_SAMPLER = '/content/drive/MyDrive/Eksperimen_Alzheimer_VGG19'
FOLDER_CLASS_WEIGHT = '/content/drive/MyDrive/Eksperimen_Alzheimer_VGG19_CW'

# Panggil untuk folder Sampler
evaluate_and_visualize_all(FOLDER_SAMPLER, "Sampler")

# Panggil untuk folder Class Weight
evaluate_and_visualize_all(FOLDER_CLASS_WEIGHT, "Class Weight")